# rHEALPix polygon polyfill

Step-by-step BFS for `polygon2rhealpix` — same multipolygon pattern as the other polyfill notebooks.

1. Seed cell at each part's bbox centroid
2. BFS via `current_cell.neighbors(plane=False)` while cells intersect the bbox
3. `check_predicate` on each visited cell against the input polygon part

Reference: [`polygon2rhealpix`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/vector2dggs/vector2rhealpix.py)

Input: [`multipolygon.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson) — **12 features**, **13 polygon parts**. **rHEALPix resolution** is shown on every frame.

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio-ffmpeg

In [ ]:
"""Step-by-step polygon2rhealpix animation (multipolygon.geojson)."""
from collections import deque
from pathlib import Path

import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import MultiPolygon, box

from vgrid.conversion.dggs2geo.rhealpix2geo import rhealpix2geo
from vgrid.conversion.dggscompact.rhealpixcompact import rhealpix_compact
from vgrid.conversion.vector2dggs.vector2rhealpix import polygon2rhealpix
from vgrid.dggs.rhealpixdggs.dggs import RHEALPixDGGS
from vgrid.utils.geometry import check_predicate

rhealpix_dggs = RHEALPixDGGS()

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson"
RESOLUTION = 8
PREDICATE = "intersects"
COMPACT = False
OUT_GIF = "polygon2rhealpix.gif"
OUT_MP4 = "polygon2rhealpix.mp4"
FRAME_EVERY_N_BFS = 8
DPI = 120
FIX_ANTIMERIDIAN = None
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e", "#00695c", "#5d4037"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polygons_from_feature(feature):
    if feature.geom_type == "Polygon":
        return [feature]
    if feature.geom_type == "MultiPolygon":
        return list(feature.geoms)
    return []


def polygons_from_gdf(gdf):
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polygons_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    parts = polygons_from_gdf(gdf)
    if not parts:
        raise ValueError("No polygon geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiPolygon(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def render_frame(
    parts,
    bbox,
    seed_poly,
    active_cells,
    title,
    path,
    resolution,
    current_poly=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    union = MultiPolygon(parts) if len(parts) > 1 else parts[0]
    minx, miny, maxx, maxy = union.bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, poly in enumerate(parts):
        color = part_color(i)
        lw = 3.0 if active_part == i else 1.8
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([poly]).plot(
            ax=ax, facecolor="none", edgecolor=color, lw=lw, alpha=alpha
        )
    if bbox is not None:
        gpd.GeoSeries([bbox]).plot(
            ax=ax, facecolor="none", edgecolor="#ff7f0e", lw=1.5, linestyle="--"
        )

    visited = list(active_cells) if active_cells else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))
    if seed_poly is not None and (
        current_poly is None
        or (seed_poly is not current_poly and not seed_poly.equals(current_poly))
    ):
        ax.add_collection(cell_patches([seed_poly], "#d62728", "#8b0000", alpha=0.7))
    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )
    ax.plot([], [], color="#ffcc00", lw=4, label="current cell")
    ax.plot([], [], color="#2ca02c", lw=4, label="bbox intersecting")
    ax.plot([], [], color="#d62728", lw=4, label="seed")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"rHEALPix resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def polygon2rhealpix_with_frames(
    parts,
    feature,
    resolution,
    predicate,
    frame_dir,
    compact=False,
    fix_antimeridian=None,
):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    merged_ids = []
    merged_polys = []
    seen_ids = set()
    accumulated = []

    def snap(title, bbox=None, seed=None, active=None, current=None, active_part=None):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            bbox,
            seed,
            active if active is not None else accumulated,
            title,
            p,
            resolution,
            current_poly=current,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    n_parts = len(parts)
    snap(
        f"1. Input res {resolution} ({n_parts} polygon part{'s' if n_parts != 1 else ''})"
    )
    snap("2. All parts (distinct colors)")

    for part_i, polygon in enumerate(parts, start=1):
        minx, miny, maxx, maxy = polygon.bounds
        bbox = box(minx, miny, maxx, maxy)
        seed_point = (bbox.centroid.x, bbox.centroid.y)
        seed_cell = rhealpix_dggs.cell_from_point(resolution, seed_point, plane=False)
        seed_cell_id = str(seed_cell)
        seed_poly = rhealpix2geo(seed_cell_id, fix_antimeridian=fix_antimeridian)
        part_idx = part_i - 1

        snap(
            f"3. Part {part_i}/{n_parts}: seed at bbox centroid",
            bbox=bbox,
            seed=seed_poly,
            active_part=part_idx,
        )

        neighbor_polys = []
        for _, neighbor in seed_cell.neighbors(plane=False).items():
            npoly = rhealpix2geo(str(neighbor), fix_antimeridian=fix_antimeridian)
            if npoly is not None and not npoly.is_empty:
                neighbor_polys.append(npoly)
        snap(
            f"3b. Part {part_i} seed neighbors ({len(neighbor_polys)} cells)",
            bbox=bbox,
            seed=seed_poly,
            active=neighbor_polys,
            active_part=part_idx,
        )

        if seed_poly.contains(bbox):
            if seed_cell_id not in seen_ids:
                seen_ids.add(seed_cell_id)
                merged_ids.append(seed_cell_id)
                merged_polys.append(seed_poly)
                accumulated.append(seed_poly)
            snap(
                f"4. Part {part_i} seed covers bbox — single cell",
                bbox=bbox,
                seed=seed_poly,
                active=[seed_poly],
                active_part=part_idx,
            )
            continue

        intersecting = {}
        covered = set()
        queue = deque([seed_cell])
        bfs_step = 0

        while queue:
            current_cell = queue.popleft()
            current_cell_id = str(current_cell)
            if current_cell_id in covered:
                continue
            covered.add(current_cell_id)

            cell_poly = rhealpix2geo(current_cell_id, fix_antimeridian=fix_antimeridian)
            if cell_poly is None or cell_poly.is_empty:
                continue
            if not cell_poly.intersects(bbox):
                continue

            intersecting[current_cell_id] = cell_poly
            for _, neighbor in current_cell.neighbors(plane=False).items():
                neighbor_id = str(neighbor)
                if neighbor_id not in covered:
                    queue.append(neighbor)

            bfs_step += 1
            if bfs_step % FRAME_EVERY_N_BFS == 0:
                snap(
                    f"4. Part {part_i} BFS step {bfs_step}: {current_cell_id} ({len(intersecting)} cells)",
                    bbox=bbox,
                    seed=seed_poly,
                    active=list(intersecting.values()),
                    current=cell_poly,
                    active_part=part_idx,
                )

        snap(
            f"5. Part {part_i} BFS complete ({len(intersecting)} bbox hits)",
            bbox=bbox,
            seed=seed_poly,
            active=list(intersecting.values()),
            active_part=part_idx,
        )

        part_final = []
        for cell_id in covered:
            cell_poly = rhealpix2geo(cell_id, fix_antimeridian=fix_antimeridian)
            if cell_poly is None or cell_poly.is_empty:
                continue
            if check_predicate(cell_poly, polygon, predicate):
                part_final.append((cell_id, cell_poly))
                if cell_id not in seen_ids:
                    seen_ids.add(cell_id)
                    merged_ids.append(cell_id)
                    merged_polys.append(cell_poly)
                    accumulated.append(cell_poly)
        snap(
            f"6. Part {part_i} after predicate '{predicate}' ({len(part_final)} cells)",
            bbox=bbox,
            seed=seed_poly,
            active=[p for _, p in part_final],
            active_part=part_idx,
        )

    final_ids = merged_ids
    final_polys = merged_polys
    if compact and merged_ids:
        final_ids = rhealpix_compact(merged_ids)
        final_polys = []
        for cell_id in final_ids:
            cell_poly = rhealpix2geo(cell_id, fix_antimeridian=fix_antimeridian)
            if cell_poly is not None and not cell_poly.is_empty:
                final_polys.append(cell_poly)
        snap(
            f"7. After rhealpix_compact ({len(final_polys)} cells)",
            active=final_polys,
        )
    else:
        snap(
            f"7. Merged result ({len(final_ids)} cells across {n_parts} parts)",
            active=final_polys,
        )

    from_poly = []
    for part in parts:
        rows = polygon2rhealpix(
            part,
            resolution,
            predicate=predicate,
            compact=compact,
            fix_antimeridian=fix_antimeridian,
        )
        from_poly.extend(row["rhealpix"] for row in rows)
    if set(from_poly) != set(final_ids):
        print(
            "Warning: cell set differs from polygon2rhealpix:",
            len(final_ids),
            "vs",
            len(set(from_poly)),
        )

    return frames, final_ids


def main():
    gdf = gpd.read_file(URL)
    parts = polygons_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} polygon part(s)")

    frame_dir = Path("_polygon2rhealpix_frames")
    frames, cell_ids = polygon2rhealpix_with_frames(
        parts,
        feature,
        RESOLUTION,
        PREDICATE,
        frame_dir,
        compact=COMPACT,
        fix_antimeridian=FIX_ANTIMERIDIAN,
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    print(f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_ids)} final cells)")
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()
